# Stage 2: Task-Specific AQG Training (Refactored)

**Version**: 2.0 (Clean, no code duplication)

Notebook ini melatih model IndoT5 untuk task AQG spesifik menggunakan modules yang sudah ada.

**Key Improvements**:
- ✅ No code duplication (menggunakan `task_trainer.py`)
- ✅ Clean separation: notebook = orchestration, modules = logic
- ✅ Bug fix applied: `label_pad_token_id=-100`
- ✅ Simplified from ~200 lines → ~50 lines

**Expected Results**:
- Training loss: 39 → 2-5
- BLEU-4: 0.005 → 0.35-0.45
- ROUGE-L: 0.0 → 0.30-0.40
- Training time: ~1-2 jam pada T4 GPU

## 1. Setup Environment

In [1]:
# Install dependencies
!pip install -q transformers peft datasets accelerate
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.3 MB/s eta 0:00:00
✓ Dependencies installed


In [2]:
# Cek versi semua library yang terinstall
import importlib
import subprocess
import sys, torch, platform

libs = [
    "transformers",
    "peft", 
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]


print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"\n ")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

# Cek Python version
import sys
print(f"\n  {'python':<20} {sys.version.split()[0]}")

# Cek CUDA
import torch
print(f"  {'cuda available':<20} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")


Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS:      Linux
Torch:   2.10.0+cu128
CUDA:    True

 
=== Library Versions ===
  transformers         5.0.0
  peft                 0.18.1
  datasets             4.0.0
  accelerate           1.13.0
  evaluate             0.4.6
  torch                2.10.0+cu128
  tokenizers           0.22.2
  rouge_score          unknown
  bert_score           0.3.12

  python               3.12.13
  cuda available       True
  cuda version         12.8
  gpu name             Tesla T4


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive mounted')

Mounted at /content/drive
✓ Google Drive mounted


In [4]:
# Setup paths and extract source code
import os, sys, zipfile, shutil

DRIVE_ROOT = '/content/drive/MyDrive/dataset_aqg'
sys.path.insert(0, '/content')

# Extract src if not exists
if not os.path.exists('/content/src'):
    shutil.copy(f'{DRIVE_ROOT}/src_finetuned.zip', '/content/src_finetuned.zip')
    with zipfile.ZipFile('/content/src_finetuned.zip', 'r') as z:
        z.extractall('/content/')
    print('✓ src extracted')
else:
    print('✓ src already exists')

print(f'✓ DRIVE_ROOT: {DRIVE_ROOT}')
print(f'✓ sys.path[0]: {sys.path[0]}')

✓ src extracted
✓ DRIVE_ROOT: /content/drive/MyDrive/dataset_aqg
✓ sys.path[0]: /content


In [5]:
# Verify GPU availability
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available! Go to Runtime > Change runtime type > T4 GPU')

print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

✓ GPU: Tesla T4
✓ Memory: 15.6 GB


## 2. Load Model with LoRA

**Using**: `src.finetuned.utils.model_loader` (no code duplication!)

In [6]:
from src.finetuned.utils.model_loader import load_model_with_lora, print_model_info

# Load model with LoRA - UPDATED: Using IndoT5 (580M params) instead of IndoNanoT5 (248M)
# IndoNanoT5 was insufficient for complex AQG task
peft_model, tokenizer = load_model_with_lora(
    model_name='Wikidepia/IndoT5-base',  # Correct model name (not LazarusNLP)
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)

# Print detailed info
print_model_info(peft_model, tokenizer)

## 3. Load Dataset

**Using**: `src.finetuned.data.dataset_loader` (reusable module)

In [7]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()
TASK_DIR = '/content/dataset_aqg/dataset-task-spesifc/'

# Copy dataset from Drive if needed
if not os.path.exists(TASK_DIR + 'train.jsonl'):
    drive_task = f'{DRIVE_ROOT}/dataset-task-spesifc'
    os.makedirs(TASK_DIR, exist_ok=True)
    for f in ['train.jsonl', 'validation.jsonl', 'test.jsonl']:
        shutil.copy(f'{drive_task}/{f}', f'{TASK_DIR}{f}')
    print('✓ Dataset copied from Drive')

# Load datasets
train_dataset = loader.load_dataset(TASK_DIR, split='train')
val_dataset = loader.load_dataset(TASK_DIR, split='validation')
test_dataset = loader.load_dataset(TASK_DIR, split='test')

print(f'\nDataset loaded:')
print(f'  Train: {len(train_dataset)} samples')
print(f'  Val:   {len(val_dataset)} samples')
print(f'  Test:  {len(test_dataset)} samples')

In [ ]:
# Validate and preview dataset
validation_results = loader.validate_dataset(train_dataset)

sample = train_dataset[0]
print('\n=== Sample Entry ===')
print(f"Input: {sample['input']}...")
print(f"Target: {sample['target']}...")

In [9]:
# Drop metadata column (required for HuggingFace datasets)
train_dataset = train_dataset.remove_columns(['metadata'])
val_dataset = val_dataset.remove_columns(['metadata'])
test_dataset = test_dataset.remove_columns(['metadata'])

print(f'✓ Metadata dropped')
print(f'  Columns: {train_dataset.column_names}')
print(f'  Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

## 4. Baseline Evaluation (Pre-Training)

**Using**: `src.finetuned.evaluation` modules

In [10]:
from src.finetuned.evaluation.metrics_calculator import MetricsCalculator
from src.finetuned.evaluation.model_evaluator import ModelEvaluator

metrics_calc = MetricsCalculator()
evaluator = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Computing baseline metrics (10 samples)...')
baseline_metrics = evaluator.evaluate_on_test_set(
    test_dataset=val_dataset,
    num_beams=4,
    include_bertscore=False,
    max_samples=10
)

print(f"\nBaseline Metrics:")
print(f"  BLEU-4:  {baseline_metrics.get('bleu_4', 0):.4f}")
print(f"  ROUGE-L: {baseline_metrics.get('rouge_l', 0):.4f}")

## 5. Configure Training

**Using**: `src.finetuned.training.task_trainer` (with bug fix applied!)

In [11]:
from src.finetuned.training.task_trainer import TaskSpecificTrainer

CHECKPOINT_DIR = '/content/drive/MyDrive/dataset_aqg/checkpoints/aqg'

# Initialize trainer (all logic in task_trainer.py)
trainer = TaskSpecificTrainer(
    model=peft_model,
    tokenizer=tokenizer,
    output_dir=CHECKPOINT_DIR,
    max_length=512,
    metrics_calculator=metrics_calc
)

print('✓ Trainer initialized')
print(f'  Checkpoints will be saved to: {CHECKPOINT_DIR}')

## 6. Start Training

**Note**: Training menggunakan default config dari `task_trainer.py`:
- Learning rate: 1e-4
- Batch size: 32 (8×4)
- Warmup: 50 steps
- Epochs: 3
- **Bug fix applied**: `label_pad_token_id=-100` ✅

In [12]:
import time

start_time = time.time()

print('Starting task-specific AQG training...')
print('='*60)

# Train (all logic in task_trainer.py)
results = trainer.train(
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    early_stopping=True,
    early_stopping_patience=2
)

elapsed = (time.time() - start_time) / 3600
print(f'\n✓ Training completed in {elapsed:.2f} hours')
print(f'  Final training loss: {results["training_loss"]:.4f}')

## 7. Save Model & Visualize

In [13]:
# Save final model
model_path = trainer.save_final_model(output_name='indot5-python-aqg')
print(f'✓ Model saved to: {model_path}')

# Plot training curves
trainer.plot_training_curves(
    save_path=f'{CHECKPOINT_DIR}/training_curves.png'
)

## 8. Final Evaluation

In [14]:
# Re-initialize evaluator with trained model
evaluator_final = ModelEvaluator(
    model=peft_model,
    tokenizer=tokenizer,
    metrics_calculator=metrics_calc
)

print('Running comprehensive evaluation on test set...')
final_metrics = evaluator_final.evaluate_on_test_set(
    test_dataset=test_dataset,
    num_beams=4,
    include_bertscore=True,
    max_samples=None
)

print('\n=== Evaluation Results ===')
for key, value in final_metrics.items():
    print(f'{key}: {value:.4f}')

## 9. Generate Sample Outputs

In [ ]:
EVAL_DIR = '/content/drive/MyDrive/dataset_aqg/evaluation_results'

samples = evaluator_final.generate_samples(
    test_dataset=test_dataset,
    num_samples=5,
    num_beams=4,
    save_path=f'{EVAL_DIR}/sample_outputs.json'
)

print(f'✓ {len(samples)} sample outputs generated and saved')

# Preview first sample
if samples:
    print('\n=== Sample Output ===')
    print(f"Input: {samples[0]['input']}...")
    print(f"Target: {samples[0]['target']}...")
    print(f"Generated: {samples[0]['generated']}...")

## 10. Final Summary

In [16]:
import json
from pathlib import Path

# Compare with baseline
comparison = evaluator_final.compare_with_baseline(
    finetuned_metrics=final_metrics,
    baseline_metrics=baseline_metrics
)

# Save evaluation report
Path(EVAL_DIR).mkdir(parents=True, exist_ok=True)
report = {
    'baseline_metrics': baseline_metrics,
    'final_metrics': final_metrics,
    'comparison': comparison,
    'training_time_hours': elapsed,
    'model_path': model_path,
    'config': {
        'learning_rate': 1e-4,
        'batch_size': 32,
        'epochs': 3,
        'lora_r': 8,
        'lora_alpha': 16
    }
}

with open(f'{EVAL_DIR}/evaluation_report.json', 'w') as f:
    json.dump(report, f, indent=2, default=str)

# Print summary
print('\n' + '='*60)
print('TASK-SPECIFIC AQG TRAINING SUMMARY')
print('='*60)
print(f'Training Time: {elapsed:.2f} hours')
print(f'Model saved: {model_path}')
print(f'\nMetrics Comparison:')
print(f"  BLEU-4:       {baseline_metrics.get('bleu_4',0):.4f} → {final_metrics.get('bleu_4',0):.4f}")
print(f"  ROUGE-L:      {baseline_metrics.get('rouge_l',0):.4f} → {final_metrics.get('rouge_l',0):.4f}")
print(f"  BERTScore F1: {baseline_metrics.get('bertscore_f1',0):.4f} → {final_metrics.get('bertscore_f1',0):.4f}")

bleu_improvement = comparison.get('bleu_4_improvement_pct', 0)
print(f'\nBLEU-4 Improvement: {bleu_improvement:+.1f}%')

if final_metrics.get('bleu_4', 0) >= 0.35:
    print('\n✓ SUCCESS: BLEU-4 target achieved (>= 0.35)')
else:
    print(f"\n⚠ BLEU-4 = {final_metrics.get('bleu_4',0):.4f} (target: >= 0.35)")
    print('  Consider: more epochs, lower lr, or larger dataset')

print('\n✓ Fine-tuning pipeline complete!')
print(f'  Evaluation report: {EVAL_DIR}/evaluation_report.json')
print(f'  Sample outputs: {EVAL_DIR}/sample_outputs.json')